# SFO Domestic Flight-Delay Prediction V2

**Approach:** Predict a departure-delay distribution at **T-2 hours** and **T-4 hours** for the eight carriers departing SFO:
1. Alaska (AS)
2. American (AA)
3. Delta (DL)
4. Frontier (F9)
5. Hawaiian (HA)
6. JetBlue (B6)
7. Southwest (WN)
8. United (UA)

The delay bins are:
- < 15 mins
- 15-59 mins
- 60-119 mins
- 120-239 mins
- 240+ mins

The bins are intentionally unequal: **under 15 minutes is on time** under the BTS definition, while the longer bins reflect meaningfully different levels of passenger disruption.

Cancellations remain a separate rare-event target and should be treated as their own binary modeling task if there is time.

### Main features

- **Schedule:** carrier, destination, planned times, distance, calendar, and planned SFO volume.
- **Historical BTS:** prior carrier, destination, route, and delay-cause rates.
- **Recent operations:** SFO-wide and carrier-specific activity observed before cutoff.
- **SFO weather:** the latest SFO observation available before cutoff.
- **Same-tail lineage:** the assigned aircraft's prior schedule and any prior actual event already observed by cutoff.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

CONFIG_PATH = ROOT / "config" / "project_config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

CONFIG["features"]

{'weather_tolerance_hours': 1,
 'historical_reliability_lookback_days': [90, 365],
 'recent_airport_operations_windows_hours': [1, 3, 6],
 'carrier_route_windows_hours': [6, 24]}

In [2]:
pd.DataFrame({
    "bin": CONFIG["bin_labels"],
    "minutes": ["<15", "15-59", "60-119", "120-239", "240+"],
    "passenger meaning": ["on time", "minor delay", "moderate delay", "major delay", "severe delay"],
})

,bin,minutes,passenger meaning
0,under_15,<15,on time
1,15_to_under_60,15-59,minor delay
2,60_to_under_120,60-119,moderate delay
3,120_to_under_240,120-239,major delay
4,240_plus,240+,severe delay


<div style="background:#0b1220;color:#e5eefc;border:1px solid #1f2a44;border-radius:12px;padding:18px 18px 12px 18px;max-width:860px;">
  <div style="font-weight:700;font-size:18px;margin-bottom:10px;color:#dbeafe;">Expanding Temporal CV</div>
  <svg width="820" height="210" viewBox="0 0 820 210" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Expanding temporal cross-validation folds">
    <rect x="0" y="0" width="820" height="210" fill="#0b1220"/>
    <g fill="#dbeafe" font-family="Arial, sans-serif" font-size="16" font-weight="600">
      <text x="95" y="28">2022</text>
      <text x="245" y="28">2023</text>
      <text x="395" y="28">2024</text>
      <text x="545" y="28">2025</text>
      <text x="665" y="28">Jan-May 2026</text>
    </g>
    <g fill="#94a3b8" font-family="Arial, sans-serif" font-size="15" font-weight="600">
      <text x="10" y="63">Fold 1</text>
      <text x="10" y="98">Fold 2</text>
      <text x="10" y="133">Fold 3</text>
      <text x="10" y="178">Forward</text>
    </g>
    <g>
      <rect x="85" y="42" width="140" height="24" rx="6" fill="#3b82f6"/>
      <rect x="225" y="42" width="140" height="24" rx="6" fill="#f59e0b"/>
      <rect x="225" y="77" width="140" height="24" rx="6" fill="#3b82f6"/>
      <rect x="365" y="77" width="140" height="24" rx="6" fill="#f59e0b"/>
      <rect x="365" y="112" width="140" height="24" rx="6" fill="#3b82f6"/>
      <rect x="505" y="112" width="140" height="24" rx="6" fill="#f59e0b"/>
      <rect x="85" y="157" width="560" height="24" rx="6" fill="#3b82f6"/>
      <rect x="645" y="157" width="145" height="24" rx="6" fill="#14b8a6"/>
    </g>
    <g fill="#eff6ff" font-family="Arial, sans-serif" font-size="14" font-weight="700">
      <text x="138" y="58">Train</text>
      <text x="265" y="58">Validate</text>
      <text x="278" y="93">Train</text>
      <text x="405" y="93">Validate</text>
      <text x="418" y="128">Train</text>
      <text x="545" y="128">Validate</text>
      <text x="340" y="173">Train</text>
      <text x="700" y="173">Test</text>
    </g>
  </svg>
</div>

We always train on the past and validate on the next year. That keeps the time order realistic and prevents future information from leaking backward.

<div style="background:#0b1220;color:#e5eefc;border:1px solid #1f2a44;border-radius:12px;padding:18px 18px 12px 18px;max-width:860px;">
  <div style="font-weight:700;font-size:18px;margin-bottom:10px;color:#dbeafe;">Prediction cutoff at T-2</div>
  <svg width="820" height="230" viewBox="0 0 820 230" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Simple prediction cutoff timeline">
    <rect x="0" y="0" width="820" height="230" rx="12" fill="#0f172a"/>
    <text x="200" y="32" fill="#bfdbfe" font-family="Arial, sans-serif" font-size="20" font-weight="700">What is known by prediction cutoff?</text>
    <rect x="70" y="86" width="310" height="28" rx="14" fill="#1f8a70"/>
    <rect x="380" y="86" width="370" height="28" rx="14" fill="#5b3344"/>
    <line x1="80" y1="100" x2="740" y2="100" stroke="#cbd5e1" stroke-width="2"/>
    <line x1="380" y1="58" x2="380" y2="168" stroke="#fbbf24" stroke-width="4" stroke-dasharray="6 6"/>
    <circle cx="180" cy="100" r="7" fill="#7dd3fc"/>
    <circle cx="380" cy="100" r="7" fill="#fbbf24"/>
    <circle cx="598" cy="100" r="7" fill="#fb7185"/>
    <circle cx="720" cy="100" r="7" fill="#fb7185"/>
    <text x="96" y="76" fill="#a7f3d0" font-family="Arial, sans-serif" font-size="13" font-weight="700">Known by cutoff</text>
    <text x="640" y="76" fill="#fecaca" font-family="Arial, sans-serif" font-size="13" font-weight="700">Not known yet</text>
    <rect x="70" y="132" width="220" height="28" rx="8" fill="#1e293b" stroke="#334155"/>
    <text x="180" y="151" text-anchor="middle" fill="#e2e8f0" font-family="Arial, sans-serif" font-size="12" font-weight="700">Schedule, tail, history, weather</text>
    <rect x="320" y="40" width="115" height="28" rx="8" fill="#1e293b" stroke="#475569"/>
    <text x="380" y="59" text-anchor="middle" fill="#fbbf24" font-family="Arial, sans-serif" font-size="12" font-weight="700">Predict here</text>
    <rect x="542" y="132" width="112" height="28" rx="8" fill="#1e293b" stroke="#334155"/>
    <text x="598" y="151" text-anchor="middle" fill="#ffe4e6" font-family="Arial, sans-serif" font-size="12" font-weight="700">Later SFO weather</text>
    <rect x="664" y="132" width="112" height="28" rx="8" fill="#1e293b" stroke="#334155"/>
    <text x="720" y="151" text-anchor="middle" fill="#ffe4e6" font-family="Arial, sans-serif" font-size="12" font-weight="700">Actual departure</text>
    <text x="88" y="194" fill="#a7f3d0" font-family="Arial, sans-serif" font-size="12" font-weight="600">Use: assigned tail plus information already observed before cutoff</text>
    <text x="88" y="212" fill="#fdba74" font-family="Arial, sans-serif" font-size="12" font-weight="600">Do not use: realized events that happen after the cutoff</text>
  </svg>
</div>

Example for a T-2 prediction: if a flight is scheduled to depart at 3:00 PM, the cutoff is 1:00 PM. We assume its tail number is known at 1:00 PM. We can use that aircraft's scheduled prior leg and any actual prior event already observed, but not an arrival or departure that occurs after 1:00 PM.

### Possible Limitations

We are assuming Tail Number at **T-2** and **T-4** does not change so that we can cleanly use tail lineage. However, SFO may decide to switch carriers moments prior to departure to mitigate severe delays for passengers.

### Key Metrics

- **Accuracy:** exact-bin hit rate. Useful, but it can look strong even if the model mostly predicts `under_15`.
- **Recall:** how many true cases in a bin the model catches. This matters most for the longer-delay bins because missing them is costly.
- **Macro-F1:** balances precision and recall and gives each bin equal weight. This is the best single summary when classes are imbalanced.
- **Within-one-bin accuracy:** gives credit when the model is close even if it misses the exact bucket.
- **Log loss:** checks whether predicted probabilities make sense, not just whether the top class is right.

The goal is not raw accuracy alone. We want a model that still catches meaningful delays, not one that wins by overpredicting the on-time bin.